In [1]:
import sys
import uuid
from pathlib import Path
from datetime import datetime, timezone

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
from src.config import validate_config

from src.quickbooks import (
    get_accounts,
    get_customers,
    get_journal_entries
)

from src.azure_sql import (
    get_engine,
    test_connection,
    read_sql,
    write_dataframe
)

from src.transformations import (
    build_raw_dataframe,
    transform_accounts,
    transform_customers,
    transform_journal_entries
)

from src.qa import (
    create_check,
    check_not_empty,
    check_no_nulls,
    check_unique,
    check_journals_balanced,
    results_to_dataframe,
    raise_if_critical_failures
)

from sqlalchemy.dialects.mssql import NVARCHAR, DATETIMEOFFSET

In [3]:
validate_config()

engine = get_engine()
test_connection(engine)

batch_id = str(uuid.uuid4())
run_id = str(uuid.uuid4())
extracted_at = datetime.now(timezone.utc)

print("Pipeline started")
print("Batch ID:", batch_id)

Pipeline started
Batch ID: 4d11049d-f6cb-43c3-bc14-afed23bbb977


In [4]:
accounts = get_accounts()
customers = get_customers()

journals = get_journal_entries(
    start_date="2023-09-01",
    end_date="2026-08-31"
)

df_accounts_raw = build_raw_dataframe(
    accounts, batch_id, extracted_at
)

df_customers_raw = build_raw_dataframe(
    customers, batch_id, extracted_at
)

df_journals_raw = build_raw_dataframe(
    journals, batch_id, extracted_at
)

bronze_dtype = {
    "entity_id": NVARCHAR(100),
    "batch_id": NVARCHAR(36),
    "payload_json": NVARCHAR(None)
}

write_dataframe(
    df_accounts_raw,
    "qbo_accounts_raw",
    "bronze",
    "append",
    engine,
    bronze_dtype
)

write_dataframe(
    df_customers_raw,
    "qbo_customers_raw",
    "bronze",
    "append",
    engine,
    bronze_dtype
)

write_dataframe(
    df_journals_raw,
    "qbo_journal_entries_raw",
    "bronze",
    "append",
    engine,
    bronze_dtype
)

print("Bronze: OK")

Bronze: OK


In [5]:
dim_account = transform_accounts(df_accounts_raw)

dim_customer = transform_customers(df_customers_raw)

fact_gl = transform_journal_entries(
    df_journals_raw,
    journal_min=202309,
    journal_max=202608
)

write_dataframe(
    dim_account,
    "dim_account",
    "silver",
    "replace",
    engine
)

write_dataframe(
    dim_customer,
    "dim_customer",
    "silver",
    "replace",
    engine
)

write_dataframe(
    fact_gl,
    "fact_gl",
    "silver",
    "replace",
    engine
)

print("Silver: OK")

Silver: OK


In [6]:
gl = fact_gl.merge(
    dim_account[
        [
            "account_id",
            "account_type",
            "account_subtype",
            "classification"
        ]
    ],
    on="account_id",
    how="left"
)

pnl = gl[
    gl["classification"].isin(
        ["Revenue", "Expense"]
    )
].copy()

pnl["actual_amount"] = pnl.apply(
    lambda x:
        -x["signed_amount"]
        if x["classification"] == "Revenue"
        else x["signed_amount"],
    axis=1
)

pnl_monthly_actual = (
    pnl
    .groupby(
        [
            "year",
            "month",
            "year_month",
            "account_id",
            "account_name",
            "account_type",
            "account_subtype",
            "classification"
        ],
        dropna=False
    )["actual_amount"]
    .sum()
    .reset_index()
)

write_dataframe(
    pnl_monthly_actual,
    "pnl_monthly_actual",
    "gold",
    "replace",
    engine
)

print("Gold: OK")

Gold: OK


In [7]:
checks = []

checks.append(
    check_not_empty(
        df_accounts_raw,
        "bronze",
        "accounts_not_empty",
        run_id
    )
)

checks.append(
    check_not_empty(
        df_customers_raw,
        "bronze",
        "customers_not_empty",
        run_id
    )
)

checks.append(
    check_not_empty(
        df_journals_raw,
        "bronze",
        "journals_not_empty",
        run_id
    )
)

checks.append(
    check_no_nulls(
        fact_gl,
        "account_id",
        "silver",
        run_id
    )
)

checks.append(
    check_no_nulls(
        fact_gl,
        "txn_date",
        "silver",
        run_id
    )
)

checks.append(
    check_unique(
        fact_gl,
        ["journal_id", "line_id"],
        "silver",
        run_id
    )
)

checks.append(
    check_journals_balanced(
        fact_gl,
        run_id
    )
)

checks.append(
    create_check(
        layer="silver",
        check_name="expected_journal_count",
        passed=fact_gl["journal_no"].nunique() == 36,
        actual_value=fact_gl["journal_no"].nunique(),
        expected_value=36,
        run_id=run_id
    )
)

checks.append(
    create_check(
        layer="gold",
        check_name="expected_month_count",
        passed=pnl_monthly_actual["year_month"].nunique() == 36,
        actual_value=pnl_monthly_actual["year_month"].nunique(),
        expected_value=36,
        run_id=run_id
    )
)

In [8]:
qa_results = results_to_dataframe(checks)

qa_results["actual_value"] = (
    qa_results["actual_value"].astype(str)
)

qa_results["expected_value"] = (
    qa_results["expected_value"].astype(str)
)

qa_dtype = {
    "run_id": NVARCHAR(36),
    "checked_at": DATETIMEOFFSET(),
    "layer": NVARCHAR(50),
    "check_name": NVARCHAR(200),
    "status": NVARCHAR(20),
    "severity": NVARCHAR(20),
    "actual_value": NVARCHAR(200),
    "expected_value": NVARCHAR(200),
    "message": NVARCHAR(None)
}

write_dataframe(
    qa_results,
    "pipeline_checks",
    "qa",
    "append",
    engine,
    qa_dtype
)

display(qa_results)

,run_id,checked_at,layer,check_name,status,severity,actual_value,expected_value,message
0,c4d68aec-e553-4d60-802c-0876c0daadfc,2026-09-17 17:41:22.652210+00:00,bronze,accounts_not_empty,PASS,ERROR,89,> 0,None
1,c4d68aec-e553-4d60-802c-0876c0daadfc,2026-09-17 17:41:22.652280+00:00,bronze,customers_not_empty,PASS,ERROR,411,> 0,None
2,c4d68aec-e553-4d60-802c-0876c0daadfc,2026-09-17 17:41:22.652318+00:00,bronze,journals_not_empty,PASS,ERROR,39,> 0,None
3,c4d68aec-e553-4d60-802c-0876c0daadfc,2026-09-17 17:41:22.652740+00:00,silver,account_id_not_null,PASS,ERROR,0,0,None
4,c4d68aec-e553-4d60-802c-0876c0daadfc,2026-09-17 17:41:22.652989+00:00,silver,txn_date_not_null,PASS,ERROR,0,0,None
5,c4d68aec-e553-4d60-802c-0876c0daadfc,2026-09-17 17:41:22.653829+00:00,silver,unique_journal_id_line_id,PASS,ERROR,0,0,None
6,c4d68aec-e553-4d60-802c-0876c0daadfc,2026-09-17 17:41:22.654859+00:00,silver,journals_balanced,PASS,ERROR,0,0,None
7,c4d68aec-e553-4d60-802c-0876c0daadfc,2026-09-17 17:41:22.655236+00:00,silver,expected_journal_count,PASS,ERROR,36,36,None
8,c4d68aec-e553-4d60-802c-0876c0daadfc,2026-09-17 17:41:22.655590+00:00,gold,expected_month_count,PASS,ERROR,36,36,None


In [9]:
raise_if_critical_failures(checks)

print("Pipeline completed successfully.")
print("Batch ID:", batch_id)
print("QA Run ID:", run_id)

Pipeline completed successfully.
Batch ID: 4d11049d-f6cb-43c3-bc14-afed23bbb977
QA Run ID: c4d68aec-e553-4d60-802c-0876c0daadfc
